In [ ]:
import os 
from tqdm.notebook import tqdm
import pandas as pd 
from shutil import copy2
from concurrent.futures import ThreadPoolExecutor

BASE_PATH = './'

DR= 'Diabetic Retinopathy'

# https://www.kaggle.com/mariaherrerot/datasets -> Download all datasets, except the SUSTech-SYSU dataset

# 0 - Mild
# 1 - Moderate
# 2 - No DR
# 3 - Proliferative DR
# 4 - Severe

REMAP_LABELS = {
    0: 2,
    1: 0,
    2: 1,
    3: 4,
    4: 3, 
}

ID_TO_LABELS = {
                0: f'Mild {DR}', 
                1: f'Moderate {DR}',
                2: f'No {DR}', 
                3: f'Proliferative {DR}',
                4: f'Severe {DR}',
}

DST_FOLDER = './retina_domain'

In [13]:
ID_TO_LABELS

{0: 'Mild Diabetic Retinopathy',
 1: 'Moderate Diabetic Retinopathy',
 2: 'No Diabetic Retinopathy',
 3: 'Proliferative Diabetic Retinopathy',
 4: 'Severe Diabetic Retinopathy'}

# process messidor dataset

In [14]:
def process_messidor(train_ratio = 0.8):
    
    DOMAIN_NAME = 'messidor2'
    folder_name= 'messidor2'
    folder_path = os.path.join(BASE_PATH, folder_name)
    
    csv_name = 'messidor_data.csv' 
    data_csv = pd.read_csv(os.path.join(folder_path, csv_name))
    display(data_csv)
    data_csv['diagnosis'] = data_csv['diagnosis'].replace(REMAP_LABELS)
    display(data_csv)

    # split into train and test 
    train_size = int(len(data_csv) * train_ratio)

    # Shuffle the DataFrame
    df_shuffled = data_csv.sample(frac=1, random_state=42).reset_index(drop=True)

    # Split into train and test
    train_df = df_shuffled.iloc[:train_size]
    test_df = df_shuffled.iloc[train_size:]

    # Display shapes
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    
    # now iterate over dataset and create the folder the destination folder 

    print("Iterating over splits...")
    for current_df, split_name in zip([train_df, test_df], ['train', 'test']):
        for _, row in tqdm(current_df.iterrows(), total=current_df.shape[0], desc='Processing...'):
            label_id = int(row['diagnosis'])
            label_name = ID_TO_LABELS[label_id].replace(' ', "_")
            image_name = row['id_code']

            dst_folder = os.path.join(DST_FOLDER, split_name, DOMAIN_NAME, label_name)
            # Create folder if need
            os.makedirs(dst_folder, exist_ok=True)

            src_image_path = os.path.join(folder_path, 'messidor-2', 'messidor-2', 'preprocess', image_name)
            
            copy2(src_image_path, dst_folder)
    print("Done!")

# process_messidor()

In [15]:
def process_idrid_retina():
    
    DOMAIN_NAME = 'idrid_retina'
    folder_name= 'idrid_retina'
    folder_path = os.path.join(BASE_PATH, folder_name)
    
    csv_name = 'idrid_labels.csv' 
    data_csv = pd.read_csv(os.path.join(folder_path, csv_name))
    display(data_csv)
    data_csv['diagnosis'] = data_csv['diagnosis'].replace(REMAP_LABELS)
    display(data_csv)

    # Split into train and test
    train_df = data_csv[~data_csv.id_code.str.contains('test')]
    test_df = data_csv[data_csv.id_code.str.contains('test')]

    # Display shapes
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    
    # return 0
    # now iterate over dataset and create the folder the destination folder 

    print("Iterating over splits...")
    for current_df, split_name in zip([train_df, test_df], ['train', 'test']):
        for _, row in tqdm(current_df.iterrows(), total=current_df.shape[0], desc='Processing...'):
            label_id = int(row['diagnosis'])
            label_name = ID_TO_LABELS[label_id].replace(' ', "_")
            image_name = row['id_code'] + '.jpg'

            dst_folder = os.path.join(DST_FOLDER, split_name, DOMAIN_NAME, label_name)
            # Create folder if need
            os.makedirs(dst_folder, exist_ok=True)

            src_image_path = os.path.join(folder_path, 'Imagenes', 'Imagenes', image_name)
            
            copy2(src_image_path, dst_folder)
    print("Done!")

# process_idrid_retina()

In [16]:
def process_aptos():
    
    DOMAIN_NAME = 'aptos2019'
    folder_name= 'aptos2019'
    folder_path = os.path.join(BASE_PATH, folder_name)
    

    # Split into train and test
    train_df = pd.read_csv(os.path.join(folder_path, 'train_1.csv'))
    train_df['id_code'] = 'train_images/train_images/' + train_df['id_code'] 
    
    val_df = pd.read_csv(os.path.join(folder_path, 'valid.csv'))
    val_df['id_code'] = 'val_images/val_images/' + val_df['id_code'] 

    train_df = pd.concat([train_df, val_df])
    display(train_df)
    train_df['diagnosis'] = train_df['diagnosis'].replace(REMAP_LABELS)
    display(train_df)

    test_df = pd.read_csv(os.path.join(folder_path, 'test.csv'))
    test_df['id_code'] = 'test_images/test_images/' + test_df['id_code'] 
    test_df['diagnosis'] = test_df['diagnosis'].replace(REMAP_LABELS)

    # Display shapes
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    
    # return 0
    # now iterate over dataset and create the folder the destination folder 


    print("Iterating over splits...")
    for current_df, split_name in zip([train_df, test_df], ['train', 'test']):
        for _, row in tqdm(current_df.iterrows(), total=current_df.shape[0], desc="Processing"):
            label_id = int(row['diagnosis'])
            label_name = ID_TO_LABELS[label_id].replace(' ', "_")
            image_name = row['id_code'] + '.png'

            dst_folder = os.path.join(DST_FOLDER, split_name, DOMAIN_NAME, label_name)
            # Create folder if need
            os.makedirs(dst_folder, exist_ok=True)

            src_image_path = os.path.join(folder_path, image_name)
            
            copy2(src_image_path, dst_folder)
    print("Done!")

# process_aptos()

## Process DDR Dataset

In [ ]:
def process_ddr(train_ratio=0.8):
    
    DOMAIN_NAME = 'DDR'
    folder_name= 'ddr'
    folder_path = os.path.join(BASE_PATH, folder_name)
    

    # Split into train and test
    csv_name = 'DR_grading.csv' 
    data_csv = pd.read_csv(os.path.join(folder_path, csv_name))
    data_csv['diagnosis'] = data_csv['diagnosis'].replace(REMAP_LABELS)
    display(data_csv)

    # split into train and test 
    train_size = int(len(data_csv) * train_ratio)

    # Shuffle the DataFrame
    df_shuffled = data_csv.sample(frac=1, random_state=42).reset_index(drop=True)

    # Split into train and test
    train_df = df_shuffled.iloc[:train_size]
    test_df = df_shuffled.iloc[train_size:]

    # Display shapes
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    

    def process_row(args):
        row, split_name = args

        label_id = int(row['diagnosis'])
        label_name = ID_TO_LABELS[label_id].replace(' ', "_")
        image_name = row['id_code']

        dst_folder = os.path.join(DST_FOLDER, split_name, DOMAIN_NAME, label_name)
        os.makedirs(dst_folder, exist_ok=True)

        src_image_path = os.path.join(folder_path, 'DR_grading', 'DR_grading', image_name)
        copy2(src_image_path, dst_folder)


    tasks = []
    for current_df, split_name in zip([train_df, test_df], ['train', 'test']):
        for _, row in current_df.iterrows():
            tasks.append((row, split_name))

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(
            executor.map(process_row, tasks),
            total=len(tasks),
            desc="Processing"
        ))

process_ddr()

,id_code,diagnosis
0,20170413102628830.jpg,2
1,20170413111955404.jpg,2
2,20170413112015395.jpg,2
3,20170413112017305.jpg,2
4,20170413112528859.jpg,2
...,...,...
12517,007-6761-400.jpg,3
12518,007-6762-400.jpg,3
12519,007-6763-400.jpg,3
12520,007-6764-400.jpg,3


Train shape: (10017, 2)
Test shape: (2505, 2)


Processing:   0%|          | 0/12522 [00:00<?, ?it/s]